In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

RANDOM_STATE = 42


In [2]:
app = pd.read_csv("../data/application_train.csv")
bureau = pd.read_csv("../data/bureau.csv")

print(f"Application table: {app.shape}")
print(f"Bureau table:      {bureau.shape}")

Application table: (307511, 122)
Bureau table:      (1716428, 17)


In [3]:
print("Columns:")
print(bureau.columns.tolist())
print("\nFirst few rows:")
bureau.head()

Columns:
['SK_ID_CURR', 'SK_ID_BUREAU', 'CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'DAYS_CREDIT', 'CREDIT_DAY_OVERDUE', 'DAYS_CREDIT_ENDDATE', 'DAYS_ENDDATE_FACT', 'AMT_CREDIT_MAX_OVERDUE', 'CNT_CREDIT_PROLONG', 'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_LIMIT', 'AMT_CREDIT_SUM_OVERDUE', 'CREDIT_TYPE', 'DAYS_CREDIT_UPDATE', 'AMT_ANNUITY']

First few rows:


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


In [4]:
bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    bureau_count=('SK_ID_BUREAU', 'count'),
    bureau_active_count=('CREDIT_ACTIVE', lambda x: (x == 'Active').sum()),
    bureau_closed_count=('CREDIT_ACTIVE', lambda x: (x == 'Closed').sum()),
    bureau_amt_credit_sum=('AMT_CREDIT_SUM', 'sum'),
    bureau_amt_credit_mean=('AMT_CREDIT_SUM', 'mean'),
    bureau_amt_credit_max=('AMT_CREDIT_SUM', 'max'),
    bureau_amt_debt_sum=('AMT_CREDIT_SUM_DEBT', 'sum'),
    bureau_overdue_max=('AMT_CREDIT_SUM_OVERDUE', 'max'),
    bureau_days_credit_max=('DAYS_CREDIT', 'max'),
    bureau_days_credit_min=('DAYS_CREDIT', 'min'),
)

# Debt-to-credit ratio, a derived feature
bureau_agg['bureau_debt_credit_ratio'] = (
    bureau_agg['bureau_amt_debt_sum'] / bureau_agg['bureau_amt_credit_sum'].replace(0, np.nan)
)

print(f"Bureau aggregated shape: {bureau_agg.shape}")
bureau_agg.head()

Bureau aggregated shape: (305811, 11)


,bureau_count,bureau_active_count,bureau_closed_count,bureau_amt_credit_sum,bureau_amt_credit_mean,bureau_amt_credit_max,bureau_amt_debt_sum,bureau_overdue_max,bureau_days_credit_max,bureau_days_credit_min,bureau_debt_credit_ratio
SK_ID_CURR,,,,,,,,,,,
100001,7,3,4,1453365.000,207623.571429,378000.0,596686.5,0.0,-49,-1572,0.410555
100002,8,2,6,865055.565,108131.945625,450000.0,245781.0,0.0,-103,-1437,0.284122
100003,4,1,3,1017400.500,254350.125000,810000.0,0.0,0.0,-606,-2586,0.000000
100004,2,0,2,189037.800,94518.900000,94537.8,0.0,0.0,-408,-1326,0.000000
100005,3,2,1,657126.000,219042.000000,568800.0,568408.5,0.0,-62,-373,0.864992


In [5]:
df = app.merge(bureau_agg, on='SK_ID_CURR', how='left')
print(f"Merged shape: {df.shape}")

Merged shape: (307511, 133)


In [6]:
feature_cols = [c for c in df.columns if c not in ['TARGET', 'SK_ID_CURR']]

numeric_features = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df[feature_cols].select_dtypes(include=['object']).columns.tolist()

print(f"Numeric: {len(numeric_features)}, Categorical: {len(categorical_features)}")

X = df[feature_cols]
y = df['TARGET']

Numeric: 115, Categorical: 16


C:\Users\frana\AppData\Local\Temp\ipykernel_15500\2832074999.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = df[feature_cols].select_dtypes(include=['object']).columns.tolist()


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE,
)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])
pipe = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=5000, class_weight="balanced")),
])

pipe.fit(X_train, y_train)

y_prob = pipe.predict_proba(X_test)[:, 1]
print(f"AUC-ROC: {roc_auc_score(y_test, y_prob):.4f}")
print(f"AUC-PR:  {average_precision_score(y_test, y_prob):.4f}")

AUC-ROC: 0.7529
AUC-PR:  0.2347
